In [ ]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer
)

# =====================================================
# LOAD DATA
# =====================================================

train = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

# Fill missing values
for col in ["prompt", "A", "B", "C", "D", "E"]:
    train[col] = train[col].fillna("")

# =====================================================
# LABELS
# =====================================================

label2id = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

train["labels"] = train["answer"].map(label2id)

# =====================================================
# SPLIT
# =====================================================

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    gss.split(
        train,
        groups=train["prompt"]
    )
)

train_df = train.iloc[train_idx]
val_df = train.iloc[val_idx]

print("Train:", len(train_df))
print("Validation:", len(val_df))